In [1]:
import pandas as pd

In [2]:
import numpy as np

In [3]:
df = pd.read_csv('dataset/train.csv')

In [4]:
df.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [5]:
df.isnull().sum()

Index               0
geohash             0
day                 0
timestamp           0
demand              0
RoadType          600
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      2495
Weather           797
dtype: int64

In [6]:
(df.isnull().sum() / len(df)) * 100

Index            0.000000
geohash          0.000000
day              0.000000
timestamp        0.000000
demand           0.000000
RoadType         0.776207
NumberofLanes    0.000000
LargeVehicles    0.000000
Landmarks        0.000000
Temperature      3.227726
Weather          1.031061
dtype: float64

In [7]:
df = df.drop(columns=['Index'], errors='ignore')

In [8]:
df[['Hour', 'Minute']] = df['timestamp'].str.split(':', expand=True).astype(int)

In [9]:
df['Hour_sin'] = np.sin(2 * np.pi * df['Hour'] / 24)
df['Hour_cos'] = np.cos(2 * np.pi * df['Hour'] / 24)

In [10]:
df['Minute_sin'] = np.sin(2 * np.pi * df['Minute'] / 60)
df['Minute_cos'] = np.cos(2 * np.pi * df['Minute'] / 60)

In [11]:
df['day_of_week'] = df['day'] % 7

In [12]:
df = df.drop(columns=['timestamp'])

In [13]:
df.head()

,geohash,day,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,Hour,Minute,Hour_sin,Hour_cos,Minute_sin,Minute_cos,day_of_week
0,qp02z1,48,0.048804,NaN,1,Not Allowed,No,NaN,NaN,0,0,0.0,1.0,0.0,1.0,6
1,qp02zt,48,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,0,0,0.0,1.0,0.0,1.0,6
2,qp08bj,48,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,0,0,0.0,1.0,0.0,1.0,6
3,qp08gt,48,0.003272,Residential,1,Not Allowed,No,NaN,Rainy,0,0,0.0,1.0,0.0,1.0,6
4,qp02zq,48,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,0,0,0.0,1.0,0.0,1.0,6


In [14]:
df = df.sort_values(by=['geohash', 'day', 'Hour', 'Minute']).reset_index(drop=True)

In [15]:
df['demand_lag_1'] = df.groupby('geohash')['demand'].shift(1)
df['demand_lag_4'] = df.groupby('geohash')['demand'].shift(4)

In [16]:
df['demand_rolling_mean_3'] = df.groupby('geohash')['demand'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)
df['demand_rolling_std_3'] = df.groupby('geohash')['demand'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).std()
)

In [17]:
df['is_Temp_missing'] = df['Temperature'].isna().astype(int)
df['is_Weather_missing'] = df['Weather'].isna().astype(int)
df['is_RoadType_missing'] = df['RoadType'].isna().astype(int)

In [18]:
# Human Routine Flags
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['is_morning_rush'] = df['Hour'].isin([8, 9, 10]).astype(int)
df['is_evening_rush'] = df['Hour'].isin([17, 18, 19, 20]).astype(int)

# 24-Hour Seasonality Lag (Assuming 15-min intervals = 96 steps)
df['demand_lag_24h'] = df.groupby('geohash')['demand'].shift(96)

In [19]:
df['geohash_macro_4'] = df['geohash'].str[:4]
df['geohash_macro_5'] = df['geohash'].str[:5]

categorical_cols = ['geohash', 'geohash_macro_4', 'geohash_macro_5', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

for col in categorical_cols:
    df[col] = df[col].fillna('Unknown').astype(str)

In [20]:
# Calculate mean demand for each geohash at each hour of the day
geo_hour_profile = df.groupby(['geohash', 'Hour'])['demand'].mean().reset_index()
geo_hour_profile = geo_hour_profile.rename(columns={'demand': 'historical_geo_hour_mean'})

# Merge it back into the main dataframe
df = df.merge(geo_hour_profile, on=['geohash', 'Hour'], how='left')

In [23]:
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

X = df.drop(columns=['demand', 'day'])
y = df['demand']

categorical_features = ['geohash', 'geohash_macro_4', 'geohash_macro_5', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

tscv = TimeSeriesSplit(n_splits=5)

fold_scores = []

print("Starting Time-Series Cross-Validation...")
for fold, (train_index, val_index) in enumerate(tscv.split(X)):
    print(f"\n--- Fold {fold + 1} ---")
    
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    
    model = CatBoostRegressor(
        iterations=1500,        
        learning_rate=0.03,     
        depth=8, 
        l2_leaf_reg=5,          
        cat_features=categorical_features, 
        eval_metric='RMSE',
        random_seed=42,
        verbose=100, 
        early_stopping_rounds=50 
    )
    
    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        use_best_model=True
    )
    
    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    print(f"Fold {fold + 1} RMSE: {rmse:.4f}")
    fold_scores.append(rmse)

print(f"\nAverage Cross-Validation RMSE: {np.mean(fold_scores):.4f}")

print("\nTraining final model on full dataset")
final_model = CatBoostRegressor(
    iterations=model.get_best_iteration() or 1500, 
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=5,
    cat_features=categorical_features,
    random_seed=42,
    verbose=100
)
final_model.fit(X, y)

Starting Time-Series Cross-Validation...

--- Fold 1 ---
0:	learn: 0.0994383	test: 0.1600430	best: 0.1600430 (0)	total: 68.8ms	remaining: 1m 43s
100:	learn: 0.0248183	test: 0.0444688	best: 0.0444688 (100)	total: 761ms	remaining: 10.5s
200:	learn: 0.0219713	test: 0.0381161	best: 0.0381161 (200)	total: 1.52s	remaining: 9.86s
300:	learn: 0.0207524	test: 0.0370059	best: 0.0370059 (300)	total: 2.25s	remaining: 8.97s
400:	learn: 0.0195328	test: 0.0368192	best: 0.0367698 (390)	total: 3.01s	remaining: 8.26s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.03670424566
bestIteration = 432

Shrink model to first 433 iterations.
Fold 1 RMSE: 0.0367

--- Fold 2 ---
0:	learn: 0.1319712	test: 0.0980345	best: 0.0980345 (0)	total: 13.1ms	remaining: 19.6s
100:	learn: 0.0249725	test: 0.0258374	best: 0.0258374 (100)	total: 1.23s	remaining: 17.1s
200:	learn: 0.0219136	test: 0.0239736	best: 0.0239736 (200)	total: 2.51s	remaining: 16.2s
300:	learn: 0.0208641	test: 0.0233500	best: 0.0233500

CatBoostRegressor(cat_features=['geohash', 'geohash_macro_4', 'geohash_macro_5', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather'], depth=8, iterations=1234, l2_leaf_reg=5, learning_rate=0.03, loss_function='RMSE', random_seed=42, verbose=100)

In [24]:
test_df = pd.read_csv('dataset/test.csv')

In [25]:
test_df['is_test'] = 1
df['is_test'] = 0

In [26]:
max_train_day = df['day'].max()
train_tail = df[df['day'] >= (max_train_day - 3)].copy()

In [27]:
cols_to_keep = ['Index', 'geohash', 'day', 'demand', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather']

In [28]:
test_df[['Hour', 'Minute']] = test_df['timestamp'].str.split(':', expand=True).astype(int)
test_df['Hour_sin'] = np.sin(2 * np.pi * test_df['Hour'] / 24)
test_df['Hour_cos'] = np.cos(2 * np.pi * test_df['Hour'] / 24)
test_df['Minute_sin'] = np.sin(2 * np.pi * test_df['Minute'] / 60)
test_df['Minute_cos'] = np.cos(2 * np.pi * test_df['Minute'] / 60)
test_df['day_of_week'] = test_df['day'] % 7
test_df = test_df.drop(columns=['timestamp'])

In [29]:
combined_df = pd.concat([train_tail, test_df], ignore_index=True)

In [30]:
combined_df = combined_df.sort_values(by=['geohash', 'day', 'Hour', 'Minute']).reset_index(drop=True)

In [31]:

combined_df['demand_lag_1'] = combined_df.groupby('geohash')['demand'].shift(1)
combined_df['demand_lag_4'] = combined_df.groupby('geohash')['demand'].shift(4)
combined_df['demand_rolling_mean_3'] = combined_df.groupby('geohash')['demand'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)
combined_df['demand_rolling_std_3'] = combined_df.groupby('geohash')['demand'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).std()
)

In [32]:
# Human Routine Flags for Test
combined_df['is_weekend'] = combined_df['day_of_week'].isin([5, 6]).astype(int)
combined_df['is_morning_rush'] = combined_df['Hour'].isin([8, 9, 10]).astype(int)
combined_df['is_evening_rush'] = combined_df['Hour'].isin([17, 18, 19, 20]).astype(int)

# 24-Hour Seasonality Lag for Test
combined_df['demand_lag_24h'] = combined_df.groupby('geohash')['demand'].shift(96)

In [33]:
combined_df['is_Temp_missing'] = combined_df['Temperature'].isna().astype(int)
combined_df['is_Weather_missing'] = combined_df['Weather'].isna().astype(int)
combined_df['is_RoadType_missing'] = combined_df['RoadType'].isna().astype(int)

In [34]:
combined_df['geohash_macro_4'] = combined_df['geohash'].str[:4]
combined_df['geohash_macro_5'] = combined_df['geohash'].str[:5]

categorical_cols = ['geohash', 'geohash_macro_4', 'geohash_macro_5', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
for col in categorical_cols:
    combined_df[col] = combined_df[col].fillna('Unknown').astype(str)

In [43]:
# 1. Drop the old inherited column to prevent _x / _y suffixing
if 'historical_geo_hour_mean' in combined_df.columns:
    combined_df = combined_df.drop(columns=['historical_geo_hour_mean'])

# 2. Merge the EXACT SAME training profile into the combined dataframe
combined_df = combined_df.merge(geo_hour_profile, on=['geohash', 'Hour'], how='left')

# 3. Fill any brand-new geohash/hour combinations with the global mean
global_mean = df['demand'].mean()
combined_df['historical_geo_hour_mean'] = combined_df['historical_geo_hour_mean'].fillna(global_mean)

In [45]:
final_test_df = combined_df[combined_df['is_test'] == 1].copy()

In [47]:
X_test = final_test_df.drop(columns=['demand', 'is_test', 'day', 'Index'], errors='ignore')

In [49]:
print("Generating final predictions...")
predictions = final_model.predict(X_test)

Generating final predictions...


In [51]:
submission = pd.DataFrame({
    'Index': final_test_df['Index'] if 'Index' in final_test_df.columns else final_test_df.index,
    'demand': predictions
})

In [53]:
submission['demand'] = submission['demand'].clip(lower=0)
submission.to_csv('submission.csv', index=False)
print(f"Submission saved. Total rows: {len(submission)}")

Submission saved. Total rows: 41778


In [55]:
sub = pd.read_csv('submission.csv')
sub['Index'] = sub['Index'].astype(int)
sub = sub.sort_values('Index').reset_index(drop=True)
sub.to_csv('submission.csv', index=False)